# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library. We'll review the metadata, enumerate and extract records, and perform exploratory data analysis (EDA) and basic visualization, referencing all record sets, fields, and columns by their `@id`.

### Dataset Source
- Croissant JSON-LD: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata   # This is an object, do not treat as a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's inspect the record sets included in the dataset, their `@id`s, and fields' `@id`s. This helps us to reference fields precisely for subsequent extraction and processing steps.

In [ ]:
# List all available record sets and their fields by @id
from collections.abc import Iterable

record_set_ids = []

for rs in dataset.metadata.record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields and Columns (@id):")
    for field in rs.fields:
        print(f"    Field: {field.name}, @id: {field.id}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      Column: {col.name}, @id: {col.id}")
    print("")

## 3. Data Extraction
We'll load each record set's records, referencing by `@id`. Each record set is loaded as a pandas DataFrame for analysis.

> **Reference:** All `record_set`, `field`, and `column` identifiers here are `@id`s as discovered above.

In [ ]:
# Extract records from record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id}, {len(df)} records, columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set {record_set_id}")

# For demonstration, display head of the main record set (change <record_set_id> accordingly)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    if main_record_set_id in dataframes:
        print(f"\nHead of main record set {main_record_set_id}:")
        display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field (e.g., GAD-7 total score or ages if present), filter entries above a threshold, normalize, and group by demographic fields (such as gender or education level).

Replace the placeholder `@id` references below to match the actual Field or Column `@id`s printed above.

In [ ]:
# Example: Choose a numeric field and demographic field from your dataset
# Replace these values according to your dataset info from the overview step

# Below are hypothetical values. Please adapt the string to match the @id of your actual fields.
main_record_set_id = record_set_ids[0] if record_set_ids else None

# Replace with actual '@id' from print statements above
numeric_field_id = None
group_field_id = None

# Try to auto-detect common candidate for demonstration only
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    numeric_candidates = [col for col in df.columns if 'score' in col.lower() or 'age' in col.lower()]
    group_candidates = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'village' in col.lower()]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    if group_candidates:
        group_field_id = group_candidates[0]
    print(f"Using {numeric_field_id=} and {group_field_id=}")

    # Filtering
    if numeric_field_id:
        threshold = df[numeric_field_id].mean()  # filter by mean for this example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Groupby
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped average {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())


## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare its mean by a demographic group.

Feel free to adapt the axes or fields to suit your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if group present
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant-conformant data and metadata using `mlcroissant`
- Identify record sets, fields, and work solely with their `@id`s
- Extract and analyze survey data
- Filter, normalize, and group data for basic EDA
- Visualize field distributions using histograms and boxplots

For more advanced analytics or machine learning, continue building on these steps with your selected fields.